In [1]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.timestamp_stage_writer import (
    ensure_simulation_timing_config_table,
    insert_simulation_timing_config,
    build_sensor_messages_timestamped_stage,
    validate_sensor_messages_timestamped_stage,
)


In [ ]:
SCHEMA = str("capstone")
DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

IF_EXISTS_FLAG = str("replace")
CHUNK_SIZE = int(250000)

SIMULATION_TIME_CONFIG_TABLE_NAME = str("simulation_timing_config")
SIMULATION_START_DATETIME = str("2026-03-19 08:00:00+00:00")
SIMULATION_SAMPLING_INTERVAL_SECONDS = float(1.0)

TIMESTAMPED_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_stage")
TIMESTAMPED_DESTINATION_TABLE_NAME = str("synthetic_sensor_messages_timestamped_stage")

In [18]:

engine = get_engine_from_env()


In [19]:

ensure_simulation_timing_config_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
)


'simulation_timing_config'

In [20]:

insert_simulation_timing_config(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    simulation_start_datetime=SIMULATION_START_DATETIME,
    sampling_interval_seconds=SIMULATION_SAMPLING_INTERVAL_SECONDS,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
    set_active=True,
    deactivate_existing_for_run=True,
)

print("Timing config ready.")

Timing config ready.


----

In [ ]:
timestamped_table_name = build_sensor_messages_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=TIMESTAMPED_SOURCE_TABLE_NAME,
    target_table=TIMESTAMPED_DESTINATION_TABLE_NAME,
    timing_config_table=SIMULATION_TIME_CONFIG_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    if_exists=IF_EXISTS_FLAG,
    chunk_size=CHUNK_SIZE,
)

print("Built table:", timestamped_table_name)


[chunk] 1 | source rows 0 to 499,999
[timestamp] wrote chunk 1 with 500,000 rows
[chunk] 2 | source rows 500,000 to 999,999
[timestamp] wrote chunk 2 with 500,000 rows
[chunk] 3 | source rows 1,000,000 to 1,499,999
[timestamp] wrote chunk 3 with 500,000 rows
[chunk] 4 | source rows 1,500,000 to 1,999,999
[timestamp] wrote chunk 4 with 500,000 rows
[chunk] 5 | source rows 2,000,000 to 2,499,999
[timestamp] wrote chunk 5 with 500,000 rows
[chunk] 6 | source rows 2,500,000 to 2,999,999
[timestamp] wrote chunk 6 with 500,000 rows
[chunk] 7 | source rows 3,000,000 to 3,499,999
[timestamp] wrote chunk 7 with 500,000 rows
[chunk] 8 | source rows 3,500,000 to 3,999,999
[timestamp] wrote chunk 8 with 500,000 rows
[chunk] 9 | source rows 4,000,000 to 4,499,999
[timestamp] wrote chunk 9 with 500,000 rows
[chunk] 10 | source rows 4,500,000 to 4,999,999
[timestamp] wrote chunk 10 with 500,000 rows
[chunk] 11 | source rows 5,000,000 to 5,499,999
[timestamp] wrote chunk 11 with 500,000 rows
[chunk] 1

----

In [ ]:
validation_dataframe = validate_sensor_messages_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=TIMESTAMPED_DESTINATION_TABLE_NAME,
)



In [ ]:
display(validation_dataframe)

----

In [ ]:

sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        message_sequence_index,
        sensor_name,
        sensor_index,
        sensor_value,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_sensor_messages_timestamped_stage
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 104
    """
)

display(sample_dataframe)

In [ ]:
timestamp_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT observation_timestamp) AS distinct_timestamps_in_observation,
        MIN(observation_timestamp) AS observation_timestamp_min,
        MAX(observation_timestamp) AS observation_timestamp_max
    FROM capstone.synthetic_sensor_messages_timestamped_stage
    GROUP BY observation_index
    ORDER BY observation_index
    LIMIT 10
    """
)


In [ ]:

display(timestamp_check_dataframe)